# The clue-testing oracle: correctness and cost

Standalone. Needs only `qiskit` and `qiskit-ibm-runtime` — no repository, no checkout, no
`sys.path`. Everything it measures is defined in this notebook.

A nonogram oracle built the usual way enumerates: Qiskit's `PhaseOracleGate` reduces the
clue formula over all `2^n` grids and emits one multi-controlled Z per solution, so
constructing it means solving the puzzle first. The oracle below instead reads each line a
cell at a time with a reversible run-length automaton. Its size is `O(w^2)` per line where a
pattern count is exponential in the line width `w`, and nothing in it reads a solution.

Two things a published claim about it needs:

1. **Correctness** — the oracle flips the phase of exactly the satisfying grids, and returns
   its flags and workspace to zero. Exhaustive over every basis input.
2. **Cost** — gate and qubit counts against the answer-marking oracle, across 8 transpiler
   seeds, every seed recorded.

Cell 6 is the expensive one and says why.

## 1. Requirements

`pip install qiskit qiskit-ibm-runtime` if either is missing. The device model is only used
for transpilation, so nothing here contacts IBM or spends quota.

In [ ]:
import itertools, json, math, platform, time
from pathlib import Path

import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector

try:
    from qiskit_ibm_runtime.fake_provider import FakeTorino
    BACKEND = FakeTorino()
except ImportError as exc:
    raise SystemExit("pip install qiskit-ibm-runtime") from exc

print("qiskit", qiskit.__version__, "| backend", BACKEND.name)
print(platform.platform(), "|", platform.processor() or "cpu")

## 2. The oracle

Embedded verbatim, so this notebook measures the same construction that ships. The only
changes from the module are the two project imports, replaced by the local `ValidationError`
and `var_clauses` below.

In [ ]:
"""A phase oracle that tests each clue with a reversible run-length automaton.

The other two oracles in this package enumerate. ``PhaseOracleGate`` reduces the clue
formula over all ``2^n`` grids and emits one multi-controlled Z per solution, so its size
tracks the solution count. ``clue_oracle`` emits one multi-controlled X per filling a line
allows, so its size tracks ``C(w - sum(b) + 1, k)`` per line and grows exponentially in the
line width ``w``.

This one reads a line a cell at a time. Each clue defines a deterministic automaton over
``sum(b) + k`` states, laid out as the shortest line the clue accepts:

    E = 1^b_1 0 1^b_2 0 ... 0 1^b_k

State ``p`` means ``p`` characters of ``E`` are matched. A cell either advances the state,
leaves it alone, or breaks the clue:

* ``start``  -- the next character is the first ``1`` of a block: a ``1`` advances, a ``0``
  is another leading or separating empty cell and changes nothing.
* ``inside`` -- the next character is a later ``1`` of a block: a ``1`` advances, a ``0``
  ends the block early and breaks the clue.
* ``gap``    -- the next character is the mandatory ``0`` between two blocks: a ``0``
  advances, a ``1`` runs two blocks together and breaks the clue.
* ``done``   -- every block is placed: a ``0`` is a trailing empty cell, a ``1`` is a block
  too many and breaks the clue.

The automaton is not reverse-deterministic: a ``0`` read in a ``gap`` state and a ``0`` read
in the ``start`` state after it both land on the same state, so one bit per cell is needed
to say which happened. The circuit keeps that bit, holds the count of broken rules in a
binary register instead of collapsing it to a flag, and so runs as a permutation on
``(state, count, advance bits)``. The line's flag is raised when the state register is
``done`` and the count register is zero, after which the whole scan runs backwards and
returns the state, count and advance bits to zero. One workspace therefore serves every
line in turn, and only the flags accumulate.

Size is ``O(w)`` gates per cell and ``O(w^2)`` per line, against a pattern count
exponential in ``w``, and every gate carries at most ``O(log w)`` controls. Nothing here
reads a solution: the gate pattern of a line follows from its clue and its width.
"""




class ValidationError(ValueError):
    """A clue that no line of the given width can hold."""


def var_clauses(n: int, d: int | None = None):
    """(row_vars, col_vars) cell indices for an n x d grid, left-to-right, top-to-bottom."""
    if d is None:
        d = n
    row_vars = [[row * d + col for col in range(d)] for row in range(n)]
    col_vars = [[row * d + col for row in range(n)] for col in range(d)]
    return row_vars, col_vars

#: What the automaton expects of the next cell, per state.
_START, _INSIDE, _GAP, _DONE = "start", "inside", "gap", "done"


def _template(clue: tuple[int, ...]) -> str:
    """The shortest line the clue accepts: each block, separated by one empty cell."""
    blocks = [b for b in clue if b]
    return "0".join("1" * b for b in blocks)


def _state_classes(clue: tuple[int, ...]) -> list[str]:
    """One entry per automaton state, saying what the next cell must be."""
    template = _template(clue)
    classes = []
    for position in range(len(template) + 1):
        if position == len(template):
            classes.append(_DONE)
        elif template[position] == "0":
            classes.append(_GAP)
        elif position == 0 or template[position - 1] == "0":
            classes.append(_START)
        else:
            classes.append(_INSIDE)
    return classes


def _advances(state_class: str, cell: int) -> bool:
    """Whether this cell moves the automaton on one state."""
    return state_class in (_START, _INSIDE) if cell else state_class == _GAP


def _breaks(state_class: str, cell: int) -> bool:
    """Whether this cell contradicts the clue."""
    return state_class in (_GAP, _DONE) if cell else state_class == _INSIDE


def _counter_width(width: int) -> int:
    """Bits that hold a count of broken rules without wrapping back to zero."""
    return max(1, width.bit_length())


def line_states(clue: tuple[int, ...]) -> int:
    """How many states the automaton for this clue has, at most one more than the width."""
    return len(_state_classes(clue))


def line_workspace(width: int, clue: tuple[int, ...]) -> tuple[int, int, int]:
    """``(states, advance bits, counter bits)`` one line of this width and clue needs."""
    return line_states(clue), width, _counter_width(width)


def _scan(width: int, clue: tuple[int, ...]):
    """The forward pass over one line, as a circuit on that line's qubits and workspace.

    Qubit order is the line's cells, then the state register, the advance bits, the counter
    and one qubit the broken-rule condition is written to and taken off again.
    """
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import CSwapGate, MCXGate

    classes = _state_classes(clue)
    states, advance_bits, counter_bits = line_workspace(width, clue)

    cells = list(range(width))
    state = [width + i for i in range(states)]
    advance = [width + states + i for i in range(advance_bits)]
    counter = [width + states + advance_bits + i for i in range(counter_bits)]
    condition = width + states + advance_bits + counter_bits
    qc = QuantumCircuit(condition + 1, name="line scan")

    def rule_gates(cell: int, target: int, test) -> None:
        """One gate per (state, cell value) the test accepts; the states are exclusive."""
        for position, state_class in enumerate(classes):
            for value in (0, 1):
                if test(state_class, value):
                    # Control bit 0 is the cell, bit 1 the state slot.
                    qc.append(MCXGate(2, ctrl_state=value | 0b10), [cell, state[position], target])

    qc.x(state[0])
    for step, cell in enumerate(cells):
        rule_gates(cell, advance[step], _advances)
        rule_gates(cell, condition, _breaks)
        # One more broken rule, as a ripple increment over the counter.
        for bit in range(counter_bits - 1, 0, -1):
            qc.append(MCXGate(bit + 1), [condition, *counter[:bit], counter[bit]])
        qc.cx(condition, counter[0])
        rule_gates(cell, condition, _breaks)
        # The occupied slot moves up one. Running down the register visits it once.
        for position in range(states - 2, -1, -1):
            qc.append(CSwapGate(), [advance[step], state[position], state[position + 1]])
    return qc


def _line_block(width: int, clue: tuple[int, ...]):
    """One line's flag, computed and the workspace cleared again.

    Qubit order is the line's cells, its flag, then the workspace ``_scan`` takes. The block
    is its own inverse, so applying it a second time lowers the flag.
    """
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import MCXGate

    if len(_template(clue)) > width:
        shortest = len(_template(clue))
        raise ValidationError(
            f"clue {list(clue)} needs at least {shortest} cells but the line is {width} long."
        )

    states, advance_bits, counter_bits = line_workspace(width, clue)
    scan = _scan(width, clue)
    qc = QuantumCircuit(scan.num_qubits + 1, name="line automaton")

    # The flag sits between the cells and the workspace.
    scan_qubits = [*range(width), *range(width + 1, qc.num_qubits)]
    state_end = width + 1 + states - 1
    counter = [width + 1 + states + advance_bits + i for i in range(counter_bits)]

    qc.compose(scan, qubits=scan_qubits, inplace=True)
    # The clue holds when every block is placed and the broken-rule count is zero: control
    # bit 0 is the last state slot, the rest are the counter's bits at zero.
    qc.append(MCXGate(1 + counter_bits, ctrl_state=1), [state_end, *counter, width])
    qc.compose(scan.inverse(), qubits=scan_qubits, inplace=True)
    return qc


def _lines(
    row_clues: list[tuple[int, ...]], col_clues: list[tuple[int, ...]]
) -> list[tuple[list[int], tuple[int, ...]]]:
    """Every row and column, as the cell indices it covers and the clue it must match."""
    row_vars, col_vars = var_clauses(len(row_clues), len(col_clues))
    return [(row_vars[r], tuple(clue)) for r, clue in enumerate(row_clues)] + [
        (col_vars[c], tuple(clue)) for c, clue in enumerate(col_clues)
    ]


def automaton_oracle_layout(
    row_clues: list[tuple[int, ...]], col_clues: list[tuple[int, ...]]
) -> dict[str, list[int]]:
    """Which qubits hold the cells, the flags and each part of the shared workspace."""
    lines = _lines(row_clues, col_clues)
    cells = len(row_clues) * len(col_clues)
    sizes = [line_workspace(len(qubits), clue) for qubits, clue in lines]
    states = max(size[0] for size in sizes)
    advance_bits = max(size[1] for size in sizes)
    counter_bits = max(size[2] for size in sizes)

    start = cells + len(lines)
    layout = {
        "cells": list(range(cells)),
        "flags": list(range(cells, start)),
        "state": list(range(start, start + states)),
        "advance": list(range(start + states, start + states + advance_bits)),
        "counter": list(
            range(
                start + states + advance_bits,
                start + states + advance_bits + counter_bits,
            )
        ),
    }
    layout["condition"] = [start + states + advance_bits + counter_bits]
    return layout


def automaton_oracle_circuit(
    row_clues: list[tuple[int, ...]], col_clues: list[tuple[int, ...]]
):
    """The oracle as a circuit over the cell qubits, one flag per line and one workspace.

    Parameters
    ----------
    row_clues, col_clues : list[tuple[int, ...]]
        Block lengths per line, as ``puzzle_to_boolean`` takes them.

    Returns
    -------
    qiskit.QuantumCircuit
        ``rows * cols`` cell qubits, then one flag per line, then the workspace every line
        reuses. Flags and workspace come back as they were found, so the gate is a phase
        oracle on the cell register alone.

    Raises
    ------
    ValidationError
        If a clue needs more cells than its line has.
    """
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import ZGate

    layout = automaton_oracle_layout(row_clues, col_clues)
    lines = _lines(row_clues, col_clues)
    qc = QuantumCircuit(layout["condition"][0] + 1, name="automaton oracle")

    def mark() -> None:
        for flag, (qubits, clue) in zip(layout["flags"], lines):
            states, advance_bits, counter_bits = line_workspace(len(qubits), clue)
            qc.compose(
                _line_block(len(qubits), clue),
                qubits=[
                    *qubits,
                    flag,
                    *layout["state"][:states],
                    *layout["advance"][:advance_bits],
                    *layout["counter"][:counter_bits],
                    *layout["condition"],
                ],
                inplace=True,
            )

    mark()
    qc.append(ZGate().control(len(lines) - 1), layout["flags"])
    mark()
    return qc


def automaton_oracle_gate(
    row_clues: list[tuple[int, ...]], col_clues: list[tuple[int, ...]]
):
    """The same oracle as a single gate, for composing into a Grover operator."""
    return automaton_oracle_circuit(row_clues, col_clues).to_gate(label="automaton oracle")

## 3. Classical ground truth

A grid satisfies its clues when every row and column run-length-encodes to its own clue. An
all-empty line encodes to `(0,)`, which is the clue format a blank line carries.

In [ ]:
def rle(bits):
    """Lengths of contiguous filled runs, or (0,) for an empty line."""
    groups, count = [], 0
    for b in bits:
        if b:
            count += 1
        elif count:
            groups.append(count)
            count = 0
    if count:
        groups.append(count)
    return tuple(groups) if groups else (0,)


def grid_rows(bits, rows, cols):
    return [bits[r * cols:(r + 1) * cols] for r in range(rows)]


def satisfies(bits, rows, cols, row_clues, col_clues):
    """True when the grid matches every row and column clue."""
    grid = grid_rows(bits, rows, cols)
    for r in range(rows):
        if rle([bool(b) for b in grid[r]]) != tuple(row_clues[r]):
            return False
    for c in range(cols):
        if rle([bool(grid[r][c]) for r in range(rows)]) != tuple(col_clues[c]):
            return False
    return True


def clues_for(bits, rows, cols):
    """The clues a grid implies, for building a board whose solution is known."""
    grid = grid_rows(bits, rows, cols)
    row_clues = [rle([bool(b) for b in grid[r]]) for r in range(rows)]
    col_clues = [rle([bool(grid[r][c]) for r in range(rows)]) for c in range(cols)]
    return row_clues, col_clues


def one_solution_board(rows, cols, start=0):
    """The first grid of an rows x cols board whose clues admit it alone."""
    cells = rows * cols
    for g in range(start, 2 ** cells):
        bits = [(g >> k) & 1 for k in range(cells)]
        row_clues, col_clues = clues_for(bits, rows, cols)
        total = sum(
            satisfies([(h >> k) & 1 for k in range(cells)], rows, cols, row_clues, col_clues)
            for h in range(2 ** cells)
        )
        if total == 1:
            return bits, row_clues, col_clues
    raise ValueError(f"no single-solution board at {rows}x{cols}")


print(rle([True, False, True, True]), rle([False, False]))

## 4. The answer-marking baseline

For a board with one solution `s`, the enumerating oracle reduces to a single
multi-controlled Z wrapped in an X layer: `X^¬s · MCZ · X^¬s`. That identity is the whole
point — it is why a benchmark on this oracle reports the cost of Grover on an arbitrary
n-bit function rather than on the puzzle. Building it from the known solution here makes the
comparison explicit and costs nothing.

In [ ]:
def marking_oracle_circuit(solution):
    """Phase-flip the one satisfying grid: an X layer around a multi-controlled Z."""
    from qiskit.circuit.library import ZGate

    cells = len(solution)
    qc = QuantumCircuit(cells)
    zeros = [i for i, bit in enumerate(solution) if not bit]
    if zeros:
        qc.x(zeros)
    qc.append(ZGate().control(cells - 1), list(range(cells)))
    if zeros:
        qc.x(zeros)
    return qc

## 5. Correctness, exhaustively

The oracle uncomputes its flags and workspace, so it is a phase oracle on the cell register
alone. That makes one evolution enough to see every input at once: apply it to an even
superposition over all `2^cells` grids and read the phases off the amplitudes. Any amplitude
sitting outside the `workspace = 0` subspace means the uncomputation is wrong.

So the cost is one statevector per board, not `2^cells` of them.

Memory is `2^(total qubits) * 16` bytes: 23 qubits is 134 MB, 27 is 2.1 GB, 30 is 17 GB. The
check skips any board past `MAX_QUBITS` instead of exhausting RAM — raise it if you have the
room, and the summary reports what was skipped.

In [ ]:
MAX_QUBITS = 27
TOL = 1e-9


def check_board(rows, cols, row_clues, col_clues):
    """Compare the oracle's phase pattern against the classical check, over every grid."""
    cells = rows * cols
    circuit = automaton_oracle_circuit(row_clues, col_clues)
    total = circuit.num_qubits
    if total > MAX_QUBITS:
        return {"skipped": f"{total} qubits over the {MAX_QUBITS} cap", "qubits": total}

    probe = QuantumCircuit(total)
    probe.h(range(cells))
    probe.compose(circuit, inplace=True)

    t0 = time.perf_counter()
    amps = Statevector(probe).data
    elapsed = time.perf_counter() - t0

    expected = 1.0 / math.sqrt(2 ** cells)
    leak, mismatches, solutions = 0.0, [], 0
    for index, amp in enumerate(amps):
        grid_index, rest = index % (2 ** cells), index >> cells
        if rest:
            leak = max(leak, abs(amp))
            continue
        bits = [(grid_index >> k) & 1 for k in range(cells)]
        hit = satisfies(bits, rows, cols, row_clues, col_clues)
        solutions += hit
        want = -expected if hit else expected
        if abs(amp - want) > TOL:
            mismatches.append({"grid": grid_index, "got": str(amp), "want": want})

    return {
        "qubits": total,
        "cells": cells,
        "grids_checked": 2 ** cells,
        "solutions": solutions,
        "max_workspace_leak": leak,
        "phase_mismatches": len(mismatches),
        "first_mismatches": mismatches[:5],
        "seconds": round(elapsed, 3),
        "ok": leak <= TOL and not mismatches,
    }


SHAPES = [(2, 2), (2, 3), (3, 3), (2, 4), (2, 5), (3, 4), (2, 6)]

correctness = {}
for rows, cols in SHAPES:
    _, row_clues, col_clues = one_solution_board(rows, cols)
    correctness[f"{rows}x{cols}"] = check_board(rows, cols, row_clues, col_clues)
    print(f"{rows}x{cols}: {correctness[f'{rows}x{cols}']}")

### Boards that exercise the other transitions

A single-solution board walks one path. This sweeps many boards per shape so the `gap`,
`done` and broken-rule-counter branches all get hit, including clues no line can hold —
which must raise rather than mis-answer.

In [ ]:
BOARDS_PER_SHAPE = 16

varied = {}
for rows, cols in SHAPES[:4]:
    cells = rows * cols
    checked, rejected, failures = 0, 0, []
    for g in range(2 ** cells):
        bits = [(g >> k) & 1 for k in range(cells)]
        row_clues, col_clues = clues_for(bits, rows, cols)
        try:
            result = check_board(rows, cols, row_clues, col_clues)
        except ValidationError:
            rejected += 1
            continue
        if result.get("skipped"):
            continue
        checked += 1
        if not result["ok"]:
            failures.append({"grid": g, "result": result})
        if checked >= BOARDS_PER_SHAPE:
            break
    varied[f"{rows}x{cols}"] = {
        "boards_checked": checked, "clues_rejected": rejected, "failures": failures,
    }
    print(f"{rows}x{cols}: {checked} boards, {rejected} rejected, {len(failures)} failures")

## 6. Cost, against the answer-marking oracle

**The expensive cell.** Transpiling the answer-marking oracle grows far faster than the
automaton: at 20 cells it has been measured at around 8,291 s for one board at one seed,
against 16 s for an enumerating clue oracle. Eight seeds of that one shape is roughly 18
hours, and the wide shapes together run into days.

Two controls:

* `INCLUDE_MARKING = False` measures the automaton alone — minutes.
* `WIDE_SHAPES` is where the cost lives. Start empty, add one shape at a time.

Every seed is recorded and the summary reports a range. Selecting a seed on one metric and
printing another is the error to avoid.

In [ ]:
SEEDS = 8
SPARE = 2
OPTIMIZATION_LEVEL = 3
TWO_QUBIT_NAMES = ("ecr", "cz", "cx")

COST_SHAPES = [(2, 2), (2, 3), (3, 3), (2, 4), (2, 5)]
WIDE_SHAPES = []
INCLUDE_MARKING = True       # set False to skip the expensive baseline entirely


def padded(circuit, spare=SPARE):
    """Room for the synthesiser to borrow a qubit, which changes what it may emit."""
    wide = QuantumCircuit(circuit.num_qubits + spare)
    wide.compose(circuit, qubits=range(circuit.num_qubits), inplace=True)
    return wide


def transpile_runs(circuit):
    """One record per seed: two-qubit gate count and depth, nothing selected."""
    runs = []
    for seed in range(SEEDS):
        t0 = time.perf_counter()
        out = transpile(
            padded(circuit), backend=BACKEND,
            optimization_level=OPTIMIZATION_LEVEL, seed_transpiler=seed,
        )
        ops = out.count_ops()
        runs.append({
            "seed": seed,
            "two_qubit": sum(ops.get(name, 0) for name in TWO_QUBIT_NAMES),
            "depth": out.depth(),
            "seconds": round(time.perf_counter() - t0, 2),
        })
    return runs


def spread(runs, key):
    values = sorted(r[key] for r in runs)
    return {"min": values[0], "median": values[len(values) // 2], "max": values[-1]}


def measure_board(rows, cols, include_marking):
    solution, row_clues, col_clues = one_solution_board(rows, cols)
    record = {"shape": f"{rows}x{cols}", "cells": rows * cols}

    t0 = time.perf_counter()
    auto = automaton_oracle_circuit(row_clues, col_clues)
    record["automaton"] = {
        "qubits": auto.num_qubits,
        "build_seconds": round(time.perf_counter() - t0, 3),
        "runs": transpile_runs(auto),
    }
    record["automaton"]["two_qubit"] = spread(record["automaton"]["runs"], "two_qubit")
    record["automaton"]["depth"] = spread(record["automaton"]["runs"], "depth")

    if include_marking:
        t0 = time.perf_counter()
        mark = marking_oracle_circuit(solution)
        record["marking"] = {
            "qubits": mark.num_qubits,
            "build_seconds": round(time.perf_counter() - t0, 3),
            "runs": transpile_runs(mark),
        }
        record["marking"]["two_qubit"] = spread(record["marking"]["runs"], "two_qubit")
        record["marking"]["depth"] = spread(record["marking"]["runs"], "depth")
        record["gate_ratio"] = {
            k: round(record["automaton"]["two_qubit"][k] / max(1, record["marking"]["two_qubit"][k]), 2)
            for k in ("min", "median", "max")
        }
        record["qubit_ratio"] = round(
            record["automaton"]["qubits"] / record["marking"]["qubits"], 2
        )
    return record


cost = []
for rows, cols in COST_SHAPES + WIDE_SHAPES:
    t0 = time.perf_counter()
    row = measure_board(rows, cols, INCLUDE_MARKING)
    row["wall_seconds"] = round(time.perf_counter() - t0, 1)
    cost.append(row)
    print(f"{row['shape']}: {row['wall_seconds']}s  gates {row['automaton']['two_qubit']}"
          + (f"  ratio {row.get('gate_ratio')} at {row.get('qubit_ratio')}x qubits"
             if INCLUDE_MARKING else ""))

## 7. The summary, and the artifact

The ratio is reported as a range over every seed and board. A single figure from one seed is
not a result.

In [ ]:
artifact = {
    "kind": "automaton-oracle",
    "qiskit": qiskit.__version__,
    "backend": BACKEND.name,
    "machine": platform.platform(),
    "seeds": SEEDS,
    "optimization_level": OPTIMIZATION_LEVEL,
    "spare_qubits": SPARE,
    "include_marking": INCLUDE_MARKING,
    "max_qubits_for_correctness": MAX_QUBITS,
    "correctness": correctness,
    "correctness_varied": varied,
    "cost": cost,
}

out = Path("automaton-oracle.json")
out.write_text(json.dumps(artifact, indent=1, default=str) + "\n")

verified = [k for k, v in correctness.items() if v.get("ok")]
skipped = [k for k, v in correctness.items() if v.get("skipped")]
broke = [k for k, v in correctness.items() if not v.get("ok") and not v.get("skipped")]
failed_varied = {k: v["failures"] for k, v in varied.items() if v["failures"]}

print(f"correctness: {'PASS' if not broke and not failed_varied else 'FAIL'}")
print(f"  verified exhaustively: {verified}")
print(f"  skipped on the qubit cap: {skipped}")
if broke:
    print(f"  FAILED: {broke}")
if failed_varied:
    print(f"  FAILED on varied boards: {list(failed_varied)}")
if INCLUDE_MARKING and cost:
    lows = [r["gate_ratio"]["min"] for r in cost if "gate_ratio" in r]
    highs = [r["gate_ratio"]["max"] for r in cost if "gate_ratio" in r]
    qs = [r["qubit_ratio"] for r in cost if "qubit_ratio" in r]
    print(f"gate ratio across boards and seeds: {min(lows)}x to {max(highs)}x")
    print(f"qubit ratio: {min(qs)}x to {max(qs)}x")
print(f"wrote {out.resolve()}")

## 8. What to send back

Attach `automaton-oracle.json`, or paste the printed summary. What a published claim needs
from it:

* which shapes were verified exhaustively, with no phase mismatched and nothing left in the
  workspace;
* the gate cost and the qubit cost as ranges across the 8 seeds — neither ratio means
  anything without the other beside it;
* which shapes were skipped, and why.

If the automaton costs more than the enumerating oracles, that is the result and it gets
published as one.